
# 데이터 합치기

*   1만개 제한하여 수집할 시 is_truncated True 표시


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import glob

# 드라이브 내 폴더 경로
base_path = '/content/drive/MyDrive/크롤링 결과/'
folders = ['크롤링 완료', '크롤링 완료(1만개 제한)']

all_files = []
for folder in folders:
    path = os.path.join(base_path, folder, "*.csv")
    files = glob.glob(path)
    all_files.extend(files)
    print(f"✅ '{folder}' 폴더에서 {len(files)}개 파일을 찾았습니다.")

print(f"총 통합 대상 파일: {len(all_files)}개")

In [ ]:
import pandas as pd
import os
import glob
from datetime import datetime

# 1. 설정 및 경로
base_path = '/content/drive/MyDrive/크롤링 결과/'
folders = ['크롤링 완료', '크롤링 완료(1만개 제한)']

# 1만개/1.5만개 단위 수집 사이트 리스트
truncated_site_list = [
    '솔바위농원', '나잘 후레쉬', '뿡자네 제주귤밭', '37piece', '인파이프',
    'HK한국투자MINI', 'MIRAE CONDTRUCTION', 'Marque Cusine', 'FORUM',
    'KONG', '더온의료기', '피레티', 'CFDK', 'METEORA_PENSION',
    '빨간모자 피자', '대노라이프', 'Studio-Origin', '굿빙고', 'renew', 'cowskin'
]

all_df = []

for folder in folders:
    path = os.path.join(base_path, folder, "*.csv")
    files = glob.glob(path)
    is_limit_folder = '1만개 제한' in folder

    for file in files:
        df = None
        # 인코딩 해결 순차 시도
        for enc in ['utf-8-sig', 'cp949', 'euc-kr']:
            try:
                df = pd.read_csv(file, encoding=enc)
                break
            except:
                continue

        if df is None:
            print(f"❌ 읽기 실패: {os.path.basename(file)}")
            continue

        try:
            # 모든 값이 NaN이거나 필수값(url, title)이 없는 행 삭제
            df = df.dropna(how='all')
            if 'url' in df.columns and 'title' in df.columns:
                df = df.dropna(subset=['url', 'title'], how='any')
                # 공백만 들어있는 행 추가 필터링
                df = df[df['url'].astype(str).str.strip() != '']

            # 사이트명(site) 누락 시 파일명 기반 채우기
            if 'site' in df.columns:
                file_base_name = os.path.basename(file).split('_')[0]
                df['site'] = df['site'].fillna(file_base_name)

            # 제한 여부 플래그 추가
            if 'site' in df.columns:
                df['is_truncated'] = df['site'].apply(
                    lambda x: True if (is_limit_folder or str(x) in truncated_site_list) else False
                )

            # 사용자 지정 컬럼만 유지
            target_cols = ['site', 'title', 'date', 'view', 'url', 'is_truncated']
            df = df[[c for c in target_cols if c in df.columns]]

            all_df.append(df)
            print(f"✅ 로드 완료: {os.path.basename(file)} ({len(df):,}행)")

        except Exception as e:
            print(f"⚠️ 오류 발생 ({os.path.basename(file)}): {e}")

# 2. 전체 병합 및 최종 중복 제거
if all_df:
    final_df = pd.concat(all_df, ignore_index=True)

    # URL 기준 중복 제거
    before_count = len(final_df)
    if 'url' in final_df.columns:
        final_df = final_df.drop_duplicates(subset=['url'], keep='first')

    after_count = len(final_df)

    # 3. 파일 저장
    output_date = datetime.now().strftime("%Y%m%d")
    output_path = os.path.join(base_path, f'통합_데이터_최종_{output_date}.csv')
    final_df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print("\n" + "="*45)
    print(f"- 총 병합 행 수: {before_count:,}개")
    print(f"- 중복 제거 후 최종 행 수: {after_count:,}개")
    print(f"- 저장 경로: {output_path}")
    print("="*45)
else:
    print("❌ 통합할 데이터가 없습니다.")

# 데이터 전처리

*   조회수(view) 및 날짜(date) 컬럼 형식 정제
*   무의미한 단어 및 특수문자 제거



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import os

# 1. 경로 설정
base_path = '/content/drive/MyDrive/크롤링 결과/'
# 아까 통합 코드에서 저장한 파일명으로 업데이트
file_name = '통합_데이터_최종_20260317.csv'
output_path = os.path.join(base_path, file_name)

# 2. 데이터 로드
try:
    df_raw = pd.read_csv(output_path, encoding='utf-8-sig')
    print(f"✅ 데이터 로드 완료: {file_name}")
    print(f"📊 총 행 수: {len(df_raw):,}개")
except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다. 경로와 파일명을 확인해주세요: {output_path}")

In [ ]:
import pandas as pd
import numpy as np

# 1. 조회수(view) 정제: NaN은 0으로, 나머지는 숫자로 변환
df_clean = df_raw.copy()
df_clean['view_num'] = pd.to_numeric(df_clean['view'], errors='coerce').fillna(0)

# 2. 날짜(date) 정제: '26-02-13' -> '2026-02-13' 변환 후 날짜형으로 변경
def fix_date_format(d):
    if pd.isna(d) or str(d).strip() == "":
        return None
    d = str(d).strip()
    # 앞자리가 '26-'로 시작하는 경우 '20'을 붙여 4자리 연도로 보정
    if d.startswith('26-'):
        d = '20' + d
    return d

df_clean['date_dt'] = pd.to_datetime(df_clean['date'].apply(fix_date_format), errors='coerce')


In [ ]:
df_clean.columns

### 특수문자 전처리

In [ ]:
import re
import unicodedata

# 1. 수동 매핑 사전 (MANUAL_MAP)
# NFKC가 놓치는 특수 유니코드
MANUAL_MAP = {
    # 🅒🅞🅘🅝 (Negative Circled)
    '🅐': 'A', '🅑': 'B', '🅒': 'C', '🅓': 'D', '🅔': 'E', '🅕': 'F', '🅖': 'G',
    '🅗': 'H', '🅘': 'I', '🅙': 'J', '🅚': 'K', '🅛': 'L', '🅜': 'M', '🅝': 'N',
    '🅞': 'O', '🅟': 'P', '🅠': 'Q', '🅡': 'R', '🅢': 'S', '🅣': 'T', '🅤': 'U',
    '🅥': 'V', '🅦': 'W', '🅧': 'X', '🅨': 'Y', '🅩': 'Z',

    # ᗰᗩIᑎ / OᑎEᕼO (Canadian Syllabics & Styles)
    'ᗩ': 'A', 'ᗷ': 'B', 'ᑕ': 'C', 'ᗞ': 'D', 'ᖱ' : 'd', 'ᗴ': 'E', 'ᖴ': 'F', 'ᘜ': 'G',
    'ᕼ': 'H', 'ᒍ': 'J', 'ᒪ': 'L', 'ᗰ': 'M', 'ᑎ': 'N',
    'O': 'O', 'ᑭ': 'P', 'ᑫ': 'Q', 'ᖇ': 'R', 'S': 'S', '†': 'T', 'ᑌ': 'U',
    'ᐯ': 'V', 'ᗯ': 'W', '᙭': 'X', 'Y': 'Y', 'Ƶ': 'Z',

    # ᴏɴᴇʜᴏ / @ᴏɴᴇʜᴏ (Small Caps)
    'ᴀ': 'a', 'ʙ': 'b', 'ᴄ': 'c', 'ᴅ': 'd', 'ᴇ': 'e', 'ꜰ': 'f', 'ɢ': 'g',
    'ʜ': 'h', 'ɪ': 'i', 'ᴊ': 'j', 'ᴋ': 'k', 'ʟ': 'l', 'ᴍ': 'm', 'ɴ': 'n',
    'ᴏ': 'o', 'ᴘ': 'p', 'ǫ': 'q', 'ʀ': 'r', 'ꜱ': 's', 'ᴛ': 't', 'ᴜ': 'u',
    'ᴠ': 'v', 'ᴡ': 'w', 'x': 'x', 'ʏ': 'y', 'ᴢ': 'z',

    # ⓿❼⓿ / ⑧②④ (Circled & Specialized Numbers)
    '⓿': '0', '❶': '1', '❷': '2', '❸': '3', '❹': '4', '❺': '5', '❻': '6', '❼': '7', '❽': '8', '❾': '9',
    '➊': '1', '➋': '2', '➌': '3', '➍': '4', '➎': '5', '➏': '6', '➐': '7', '➑': '8', '➒': '9',
    '①': '1', '②': '2', '③': '3', '④': '4', '⑤': '5', '⑥': '6', '⑦': '7', '⑧': '8', '⑨': '9', '⑩': '10',
    '⁰': '0', '¹': '1', '²': '2', '³': '3', '⁴': '4', '⁵': '5', '⁶': '6', '⁷': '7', '⁸': '8', '⁹': '9',
    '⓪': '0'
}

# 2. 제거할 태그 리스트
target_tags = ['New', 'new', '공지', '날짜', 'AAA', 'SSS', '판매', '완료', '필독', '공지사항']

# 3. 통합 전처리 함수
def verify_cleaning(text, tags=target_tags):
    if not isinstance(text, str) or text.strip() == "":
        return ""

    # (1) 제어 문자 제거 및 표준 NFKC 정규화
    text = re.sub(r'[\xad\u200b\ufeff]', '', text)
    text = unicodedata.normalize('NFKC', text)

    # (2) MANUAL_MAP을 이용한 강력한 호모글리프 복원
    res = []
    for char in text:
        res.append(MANUAL_MAP.get(char, char))
    text = "".join(res)

    # (3) 블랙리스트 태그 제거
    tag_pattern = r'\b(?:' + '|'.join([re.escape(t) for t in tags]) + r')\b'
    text = re.sub(tag_pattern, '', text, flags=re.IGNORECASE)

    # (4) 특정 장식 기호 삭제 (내용물 보존)
    text = re.sub(r'[『』「」【】\[\]\(\)\{\}\<\>|]', ' ', text)

    # (5) 허용 범위 외 문자 제거 (한글/영문/숫자/@/-/_/. 허용)
    text = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣ\u1100-\u11FFa-zA-Z0-9\s@\-\._]', ' ', text)

    # (6) 아이디 조립 (문자 사이 공백 제거)
    text = re.sub(r'(?<=[a-zA-Z0-9@])\s+(?=[a-zA-Z0-9])', '', text)

    # (7) 연속 공백 통합 및 정리
    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [ ]:
# 전처리 적용
df_clean['title_cleaned'] = df_clean['title'].apply(verify_cleaning)
df_clean = df_clean[df_clean['title_cleaned'] != ""].reset_index(drop=True)

In [ ]:
#  테스트
test_cases = [
    "라이브홀덤솔루션 실시간토지노사이트 제작 (ㅌ＠ｅｖｏ５８８７)",
    "해외선물솔루션 제작 (『ㅌ』『@』『e』『v』『o』『5』『8』『8』『7』)",
    "해외선물 HTS 솔루션 제작 |ㅌ@ᵉᵛᵒ⁵⁸⁸⁷|",
    "⓿❼⓿",
    "ㅇㄷ ㅅㅇㅌ",
    "[AAA]",
    "[텔레:asdf123]",
    "𝕤𝕦𝕟𝕤𝕚𝕟",
    "𝑠𝑢𝑛𝑠𝕚𝑛",
    "🅒🅞🅘🅝🅣🅘🅝🅖🅛🅔",
    "ⓢⓤⓟⓔⓡⓑⓔⓔ⑧⑨②ⓐ",
    "𝚜𝚞𝚗𝚜𝚒𝚗",
    "OᑎEᕼO1004",
    "ᗰᗩIᑎ⑧②④",
    "@ᴏɴᴇʜᴏ1004",
    "R͝M̑K̀3̧3̢2̌.T̡O͌P",
    "트위터 ≠ ŖU̢B̽7̔4͚8̊.T͂O͑P̂"
]

for t in test_cases:
    print(f"원본: {t}")
    print(f"결과: {verify_cleaning(t)}\n")

### 키워드 사전 및 지명 사전 불러오기

*   불법 키워드 df_kw에 저장
*   지명(시군구, 읍면동) df_r에 저장



In [ ]:
# 1. 구글 계정 인증
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
import pandas as pd

creds, _ = default()
gc = gspread.authorize(creds)

# 2. 구글 시트 열기
sheet_url = 'https://docs.google.com/spreadsheets/d/1KXU__1wXg-oY1vJWxy7HkOhrZDe_v8NGxLn8MacQC9M/edit#gid=0'
doc = gc.open_by_url(sheet_url)
worksheet = doc.get_worksheet(0)

# 3. 에러 방지를 위해 get_all_values() 사용
data = worksheet.get_all_values()

# 4. 첫 번째 줄을 컬럼명으로 설정하여 데이터프레임 생성
keyword_raw = pd.DataFrame(data[1:], columns=data[0])

# 5. 컬럼명 변경
keyword_raw.columns = ['category', 'keyword_list']

# 6. 키워드 정제
cat_col = keyword_raw.columns[0]
key_col = keyword_raw.columns[1]

keyword_raw['keyword'] = keyword_raw[key_col].str.split(',')

# 7. 한 줄에 하나의 키워드만 오도록 펼치기
df_kw = keyword_raw.explode('keyword')
df_kw['keyword'] = df_kw['keyword'].str.strip()

# 8. 공백이나 빈 행 제거
df_kw['keyword'] = df_kw['keyword'].str.strip()
df_kw = df_kw[df_kw['keyword'] != ""].drop_duplicates().reset_index(drop=True)
df_kw['keyword'] = df_kw['keyword'].apply(lambda x: unicodedata.normalize('NFKC', str(x)))

# 9. 결과 확인
print("=== 키워드 매핑 테이블 ===")
display(df_kw[['category', 'keyword']].head(10))


In [ ]:
# 1. 지명 사전 불러오기
sheet_id = "13zAZkbRpdD0_BoAdY5p8FTgVhH_Kf_eWEUHY_JschsE"
gid = "1301961240"
export_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_r = pd.read_csv(export_url)
df_r['소분류'] = df_r['소분류'].apply(lambda x: unicodedata.normalize('NFKC', str(x)) if pd.notna(x) else x)

# 2. 결과 확인
print("=== 지명 매핑 테이블 ===")
display(df_r[['대분류', '소분류']].head(10))

In [ ]:
from google.colab import drive
import pandas as pd

# 1. 드라이브 마운트 및 파일 로드
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/크롤링 결과/전국행정동리스트.xlsx'

try:
    # 2. 행정동 리스트 로드 및 컬럼 고정
    df_admin = pd.read_excel(file_path, sheet_name='키워드추출기', skiprows=2)
    df_admin = df_admin.iloc[:, :4]
    df_admin.columns = ['시도', '시군', '구', '동면리']

    # 데이터 정제
    for col in df_admin.columns:
        df_admin[col] = df_admin[col].fillna('').astype(str).str.strip()

    # 중분류 정답지 생성
    df_admin['중분류_정답'] = (df_admin['시군'] + ' ' + df_admin['구']).str.strip()

    # 3. 광역 지자체 키워드 리스트 (필터링용)
    # 소분류에 아래 단어들이 있으면 '전체'로 간주합니다.
    sido_keywords = [
        '서울', '서울특별시', '경기', '경기도', '인천', '인천광역시', '강원', '강원도',
        '충북', '충청북도', '충남', '충청남도', '전북', '전라북도', '전남', '전라남도',
        '경북', '경상북도', '경남', '경상남도', '제주', '제주도', '제주특별자치도',
        '세종', '세종시', '세종특별자치시', '대구', '대구광역시', '대전', '대전광역시',
        '광주', '광주광역시', '울산', '울산광역시', '부산', '부산광역시'
    ]

    # 4. 매핑 함수
    def refine_location(row):
        sido_input = str(row['대분류']).strip()
        target_input = str(row['소분류']).strip()

        # [추가] 1단계: 소분류가 시도 명칭인 경우 (예: 경기, 서울 등)
        if target_input in sido_keywords or not target_input:
            return "전체", "전체"

        # 2단계: 해당 시도 데이터만 필터링
        admin_filtered = df_admin[df_admin['시도'].str.contains(sido_input[:2], na=False)]

        if admin_filtered.empty:
            return "확인필요", target_input

        # 3단계: 구(중분류) 매칭
        match_gu = admin_filtered[admin_filtered['구'].str.contains(target_input, na=False)]
        if not match_gu.empty:
            return match_gu.iloc[0]['중분류_정답'], "전체"

        # 4단계: 동(소분류) 매칭
        match_dong = admin_filtered[admin_filtered['동면리'].str.contains(target_input, na=False)]
        if not match_dong.empty:
            return match_dong.iloc[0]['중분류_정답'], match_dong.iloc[0]['동면리']

        return "확인필요", target_input

    # 5. 적용
    print("🔄 광역 지자체 필터링 및 데이터 정제 중...")
    temp_results = df_r.apply(refine_location, axis=1)

    df_r['중분류'] = [r[0] for r in temp_results]
    df_r['소분류_최종'] = [r[1] for r in temp_results]

    # 6. 결과 확인
    print("✅ 정제 완료!")
    display(df_r[['대분류', '소분류', '중분류', '소분류_최종']].head(30))

except Exception as e:
    print(f"❌ 오류 발생: {e}")

### 형태소 분석

*   키워드 사전에 있는 단어를 쪼개지지 않도록 하나의 단어로 저장
*   이후 명사만 남겨 nouns 컬럼에 저장



In [ ]:
# 1. 라이브러리 설치
!pip install kiwipiepy

from kiwipiepy import Kiwi
import pandas as pd
import unicodedata

from tqdm.auto import tqdm
tqdm.pandas()

# 2. Kiwi 초기화
kiwi = Kiwi()

# 사전 등록 시 정규화(NFKC) 필수 적용
chosung_regex = re.compile(r'^[ㄱ-ㅎ]+$')
english_regex = re.compile(r'^[a-zA-Z0-9]+$')

user_keywords = df_kw['keyword'].unique().tolist()

exclude_words = {'전체', '확인필요', ''}
region_source_cols = ['중분류', '소분류_최종', '소분류']
all_region_keywords = set()

for col in region_source_cols:
    if col in df_r.columns:
        unique_vals = df_r[col].dropna().unique()
        for val in unique_vals:
            val_str = str(val).strip()
            if val_str not in exclude_words:
                all_region_keywords.add(val_str)

region_keywords = list(all_region_keywords)

# 모든 키워드를 NFKC 정규화하여 중복 제거 및 정렬
combined_keywords = set()
for kw in (user_keywords + region_keywords):
    if pd.isna(kw): continue
    norm_kw = unicodedata.normalize('NFKC', str(kw).strip()).lower()
    if norm_kw:
        combined_keywords.add(norm_kw)

combined_keywords = sorted(list(combined_keywords), key=len, reverse=True)
combined_keywords_set = set(combined_keywords)

# Kiwi 사전에 등록
for word in combined_keywords:
    kiwi.add_user_word(word, 'NNP')

print(f"✅ {len(combined_keywords)}개의 정규화된 키워드가 분석기 사전에 등록되었습니다.")

space_variants = {}
for kw in (user_keywords + region_keywords):
    if pd.isna(kw): continue
    norm = unicodedata.normalize('NFKC', str(kw).strip()).lower()
    norm_nospace = norm.replace(' ', '')
    if norm != norm_nospace:  # 공백 제거 전후가 다른 것만
        space_variants[norm] = norm_nospace  # '시 알리스' → '시알리스'

print(f"공백 변형 키워드: {len(space_variants)}개")
print(dict(list(space_variants.items())[:10]))  # 샘플 확인

# 치환 함수 — 공백 변형이 확인된 키워드만 대상
def normalize_space_variants(text):
    for spaced, nospace in space_variants.items():
        if spaced in text:
            text = text.replace(spaced, nospace)
    return text

# 명사 추출 함수
def get_nouns_integrated(text):
    if not isinstance(text, str) or text.strip() == "": return ""

    text = text.lower()
    text = normalize_space_variants(text)   # ← Kiwi 분석 전에 치환
    tokens = kiwi.tokenize(text)
    extracted = []

    for t in tokens:
        # 유니코드 정규화 (NFC/NFKC 통일)
        form_norm = unicodedata.normalize('NFKC', t.form)

        # [1] 2글자 이상 명사(NNG, NNP) 추출
        if t.tag in ('NNG', 'NNP') and len(form_norm) >= 2:
            extracted.append(form_norm)

        # [2] 1글자 단어, 초성: 사전에 등록된 경우만 추출
        elif( t.tag == ( 'NNG', 'NNP') or chosung_regex.match(form_norm)) and len(form_norm) < 2:
            if form_norm in combined_keywords_set:
                extracted.append(form_norm)

        # [3] 영어/숫자 키워드: 사전에 있거나, 2글자 이상인 경우 추출
        elif t.tag == 'SL' or english_regex.match(form_norm):
            if form_norm in combined_keywords_set or len(form_norm) >= 2:
                extracted.append(form_norm)

    return " ".join(extracted)

# 명사 컬럼 생성
df_clean['nouns'] = df_clean['title_cleaned'].progress_apply(get_nouns_integrated)


In [ ]:
# 1. NaN(결측치) 개수 확인
nan_count = df_clean['nouns'].isna().sum()

# 2. 빈 문자열("") 개수 확인 (전처리 단계에서 ""로 바꿨을 수 있음)
empty_count = (df_clean['nouns'].astype(str).str.strip() == "").sum()

print(f"Nouns가 NaN인 행: {nan_count}개")
print(f"Nouns가 빈 값('')인 행: {empty_count}개")
print(f"전체 데이터 중 비어있는 비율: {((nan_count + empty_count) / len(df_clean)) * 100:.2f}%")


### 카테고리 분류

##### 나이브 베이즈 분류기로 학습 데이터 구축하기

In [ ]:
df_kw['category'].unique()

In [ ]:
df_r['대분류'].unique()

In [ ]:
import pandas as pd
import numpy as np
import random
import unicodedata
from itertools import combinations
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# 0. 설정 및 시드 고정
np.random.seed(42)
random.seed(42)
TARGET_N = 2000  # 카테고리당 생성할 학습 데이터 수

# 1. 키워드 사전 정규화 (사전 데이터 정리)
# 키워드와 카테고리가 들어있는 df_kw 기준
df_kw['keyword'] = df_kw['keyword'].apply(
    lambda x: unicodedata.normalize('NFKC', str(x).strip()).lower()
)

# 2. 학습용 레코드 생성 로직
train_records = []

for cat in df_kw['category'].unique():
    # 해당 카테고리의 키워드 추출
    cat_kws = df_kw[df_kw['category'] == cat]['keyword'].unique().tolist()
    if not cat_kws: continue

    # [Step 1] Unigram 생성: 모든 개별 키워드 포함
    samples = cat_kws.copy()

    # [Step 2] Bigram 생성: 두 키워드의 조합 (순서 무관하게 추출)
    bi_combos = []
    if len(cat_kws) >= 2:
        # 모든 가능한 2개 키워드 조합 생성
        bi_combos = [" ".join(c) for c in combinations(cat_kws, 2)]

    # [Step 3] Oversampling (부족한 양만큼 Bigram과 Unigram에서 랜덤하게 채움)
    needed = TARGET_N - len(samples)
    if needed > 0:
        pool = bi_combos if bi_combos else cat_kws
        samples += random.choices(pool, k=needed)

    # 해당 카테고리에 대해 2,000개의 개별 Row 생성
    for s in samples[:TARGET_N]:
        train_records.append({'category': cat, 'train_text': s})

# 3. 데이터프레임 변환
df_train_final = pd.DataFrame(train_records)

# 4. 나이브 베이즈 파이프라인 학습
# CountVectorizer는 문장에서 단어의 빈도를 계산합니다.
nb_model = Pipeline([
    ('vectorizer', CountVectorizer(
    token_pattern=r'(?u)\b\w+\b',  # 1글자 명사 인식
    ngram_range=(1, 2)              # bigram 학습 데이터와 일치
    )),
    ('nb', MultinomialNB())
])

nb_model.fit(X=df_train_final['train_text'], y=df_train_final['category'])

print(f"✅ 모델 학습 완료: 총 {len(df_train_final)}개의 샘플 학습됨")
print(f"📊 카테고리당 데이터 수: {TARGET_N}행")

In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. 모델 예측 및 확률 정보 추출
df_clean['nouns'] = df_clean['nouns'].fillna("").apply(
    lambda x: unicodedata.normalize('NFKC', str(x).strip()).lower()
)
# ---------------------------------------------------------

# 전체 카테고리 목록 및 확률 행렬 가져오기
classes = nb_model.classes_
prob_matrix = nb_model.predict_proba(df_clean['nouns'])

# Top 1, Top 2 인덱스 계산 (확률 내림차순 정렬)
top_indices = np.argsort(-prob_matrix, axis=1)

# 데이터프레임에 Top 1, 2 카테고리와 확신도 저장
df_clean['top1_cat'] = [classes[i[0]] for i in top_indices]
df_clean['top1_conf'] = np.max(prob_matrix, axis=1)
df_clean['top2_cat'] = [classes[i[1]] for i in top_indices]
df_clean['top2_conf'] = [prob_matrix[row_idx, i[1]] for row_idx, i in enumerate(top_indices)]

# ---------------------------------------------------------
# 2. 키워드 매칭 상세 정보 생성 (matched_details)
# ---------------------------------------------------------
all_kw_dict = df_kw.groupby('category')['keyword'].apply(set).to_dict()

def summarize_keywords(noun_text):
    if not noun_text: return ""
    words = set(str(noun_text).split())
    found = []
    for cat, kws in all_kw_dict.items():
        intersect = words.intersection(kws)
        if intersect:
            found.append(f"{cat}:({', '.join(intersect)})")
    return " | ".join(found)

df_clean['matched_details'] = df_clean['nouns'].apply(summarize_keywords)

# ---------------------------------------------------------
# 3. 신뢰도 계층(Logic Tier) 기반 최종 분류 함수
# ---------------------------------------------------------
def resolve_with_hierarchy(row):
    details = row['matched_details']
    main_cat = "미분류"
    tier = "Tier 4"  # 기본값: 미분류
    has_contact = 0

    # [A] 연락수단 키워드 별도 체크 (다중 라벨용)
    if "연락수단(텔레그램)" in details:
        has_contact = 1

    # [B] 카테고리별 매칭된 키워드 개수 계산 (연락수단 제외)
    cat_counts = {}
    if details:
        for item in details.split(' | '):
            parts = item.split(':(')
            cat_name = parts[0]
            if cat_name == '연락수단':
                continue
            words = parts[1].replace(')', '').split(', ')
            cat_counts[cat_name] = len(words)

    # [C] 계층별 분류 로직 적용
    # Tier 1 & 2: 키워드 기반 (사전 기반 확정)
    if cat_counts:
        max_val = max(cat_counts.values())
        winners = [cat for cat, count in cat_counts.items() if count == max_val]

        if len(winners) == 1:
            # Tier 1: 단일 카테고리 키워드가 압도적일 때
            main_cat = winners[0]
            tier = "Tier 1"
        else:
            # Tier 2: 키워드 개수 동점 시 모델 예측값으로 결정
            tier = "Tier 2"
            if row['top1_cat'] in winners:
                main_cat = row['top1_cat']
            else:
                main_cat = winners[0]

    # Tier 3: 모델 기반 (키워드가 없을 때)
    else:
        margin = row['top1_conf'] - row['top2_conf']

        # 조건: 확신도 0.25 또는 격차 0.10
        top1_cat = row['top1_cat']
        if top1_cat == '연락수단(텔레그램)':
            top1_cat = row['top2_cat']
            has_contact = 1

        if row['top1_conf'] >= 0.25:
            main_cat = row['top1_cat']
            tier = "Tier 3"
        elif row['top1_conf'] >= 0.18 and margin >= 0.10:
            main_cat = row['top1_cat']
            tier = "Tier 3 (Margin)"

    # top2도 연락수단인 경우 Tier 4
    if main_cat == '연락수단(텔레그램)':
        has_contact = 1
        top2 = row['top2_cat']
        if top2 != '연락수단(텔레그램)':
            main_cat = top2
            tier = tier + " (연락수단→top2)"
        else:
            main_cat = "미분류"
            tier = "Tier 4"

    return pd.Series([main_cat, has_contact, tier])

# ---------------------------------------------------------
# 4. 최종 적용 및 데이터 정리
# ---------------------------------------------------------
# 계층 로직 적용
df_clean[['final_category', 'is_contact', 'logic_tier']] = df_clean.apply(resolve_with_hierarchy, axis=1)

# 원-핫 인코딩 (기존 cat_ 컬럼들과 충돌 피하기 위해 재추출)
df_cat_dummies = pd.get_dummies(df_clean['final_category'], prefix='cat').astype(int)

# 기존에 존재하던 cat_ 컬럼들 제거 후 새로 합치기 (중복 방지)
other_cols = [c for c in df_clean.columns if not c.startswith('cat_')]
df_clean = pd.concat([df_clean[other_cols], df_cat_dummies], axis=1)

# 연락수단은 별도 컬럼으로 강제 적용 (다중 라벨)
df_clean['cat_연락수단'] = df_clean['is_contact']

print(f"분류 완료 (총 {len(df_clean)}행)")
display(df_clean[['title', 'final_category', 'logic_tier', 'top1_conf', 'matched_details']].head(10))

In [ ]:
# 현재 연락수단이 final_category로 된 케이스 확인
contact_as_main = df_clean[df_clean['final_category'] == '연락수단(텔레그램)']
print(f"연락수단이 메인 카테고리인 케이스: {len(contact_as_main):,}개")

In [ ]:
display(df_clean[['title', 'final_category', 'logic_tier', 'top1_conf', 'matched_details', 'is_contact']].head(50))

In [ ]:
from collections import Counter
import re

# 기존 키워드 사전 로드
existing_kw = df_kw.groupby('category')['keyword'].apply(set).to_dict()
all_existing_kw = set(kw for kws in existing_kw.values() for kw in kws)

# -----------------------------------------------
# 관점 1: Tier 4 (미분류) 에서 자주 나오는 명사
# → "이 단어들이 사전에 없어서 미분류된 것"
# -----------------------------------------------
tier4 = df_clean[df_clean['logic_tier'] == 'Tier 4']

tier4_words = Counter()
for text in tier4['nouns'].dropna():
    for word in str(text).split():
        if word not in all_existing_kw and len(word) >= 2:
            tier4_words[word] += 1

print("=" * 60)
print("📌 관점 1: Tier 4에서 자주 나오는 미사전 단어 (상위 50개)")
print("=" * 60)
tier4_df = pd.DataFrame(tier4_words.most_common(50),
                         columns=['단어', '빈도'])
display(tier4_df)

# -----------------------------------------------
# 관점 2: 오분류 의심 샘플
# → top1 카테고리 != final_category 인 케이스
# → 나이브베이즈와 키워드 판단이 엇갈린 것
# -----------------------------------------------
mismatch = df_clean[
    (df_clean['top1_cat'] != df_clean['final_category']) &
    (df_clean['logic_tier'].isin(['Tier 2', 'Tier 3', 'Tier 3 (Margin)']))
]

print("\n" + "=" * 60)
print("📌 관점 2: 오분류 의심 샘플 (상위 30개)")
print("=" * 60)
display(mismatch[['site', 'title', 'url', 'title_cleaned', 'final_category',
                   'top1_cat', 'top1_conf',
                   'matched_details', 'logic_tier']].head(30))

# -----------------------------------------------
# 관점 3: 카테고리별 미사전 단어
# → 이미 올바르게 분류된 샘플에서
#   사전에 없는 단어 추출 (보강 후보)
# -----------------------------------------------
print("\n" + "=" * 60)
print("📌 관점 3: 카테고리별 미사전 단어 TOP 20")
print("=" * 60)

for cat in ['금융', '마약', '유흥', '도박', '약물', '찌라시', '디비']:
    cat_df = df_clean[df_clean['final_category'] == cat]
    cat_words = Counter()
    for text in cat_df['nouns'].dropna():
        for word in str(text).split():
            if word not in all_existing_kw and len(word) >= 2:
                cat_words[word] += 1

    top_words = cat_words.most_common(20)
    print(f"\n[{cat}] 미사전 단어 TOP 20:")
    print(', '.join([f"{w}({c})" for w, c in top_words]))

# -----------------------------------------------
# 관점 4: 연락수단인데 다른 카테고리로 분류된 샘플
# → is_contact=1 이지만 final_category != 연락수단
# → 이 샘플들의 미사전 단어 확인
# -----------------------------------------------
print("\n" + "=" * 60)
print("📌 관점 4: 텔레그램 포함인데 금융/마약 등으로 분류된 샘플")
print("=" * 60)
contact_wrong = df_clean[
    (df_clean['is_contact'] == 1) &
    (df_clean['final_category'] != '연락수단(텔레그램)')
]
print(f"총 {len(contact_wrong):,}개\n")
print(f"카테고리 분포:")
display(contact_wrong['final_category'].value_counts().head(10))
display(contact_wrong[['site', 'title', 'title_cleaned', 'final_category',
                        'matched_details']].head(20))


In [ ]:
tier4 = df_clean[df_clean['logic_tier'] == 'Tier 4']

# 1. nouns가 비어있는지 확인
print("nouns 비어있는 비율:", (tier4['nouns'] == '').sum(), "개")
print("nouns NaN 비율:", tier4['nouns'].isna().sum(), "개")

# 2. 텍스트 길이 분포 확인
tier4['noun_len'] = tier4['nouns'].apply(lambda x: len(str(x).split()))
print(tier4['noun_len'].describe())
print("명사 0개:", (tier4['noun_len'] == 0).sum(), "개")
print("명사 1개:", (tier4['noun_len'] == 1).sum(), "개")
print("명사 2개:", (tier4['noun_len'] == 2).sum(), "개")

In [ ]:
df_clean.columns

In [ ]:
# 1. 카테고리별 건수와 비중 계산
summary_table = df_clean['final_category'].value_counts().reset_index()
summary_table.columns = ['카테고리', '건수']

# 2. 비중(%) 계산 (소수점 둘째 자리까지)
total_count = len(df_clean)
summary_table['비중(%)'] = (summary_table['건수'] / total_count * 100).round(2)

# 3. 결과 출력
print(f"📈 전체 분석 데이터 수: {total_count:,}건")
print("-" * 50)
display(summary_table)

##### train/val/test set 분류

In [ ]:
from sklearn.model_selection import train_test_split

# Tier 4 제외하고 먼저 분할
df_labeled = df_clean[df_clean['logic_tier'] != 'Tier 4'].copy()

# 카테고리 → 숫자 인코딩
categories = sorted(df_labeled['final_category'].unique())
cat2id = {cat: i for i, cat in enumerate(categories)}
id2cat = {i: cat for cat, i in cat2id.items()}

df_labeled['label'] = df_labeled['final_category'].map(cat2id)

print(f"카테고리 수: {len(categories)}")
print(f"카테고리 목록: {categories}")

# Stratified Split (8:1:1)
df_train, df_temp = train_test_split(
    df_labeled[['title_cleaned', 'label', 'logic_tier']],
    test_size=0.2,
    random_state=42,
    stratify=df_labeled['label']
)
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.5,
    random_state=42,
    stratify=df_temp['label']
)

print(f"Train : {len(df_train):,}개")
print(f"Val   : {len(df_val):,}개")
print(f"Test  : {len(df_test):,}개")

##### KLUE-RoBERTa-base 사용

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_PATH = '/content/drive/MyDrive/illegal_ad_classifier/'
os.makedirs(SAVE_PATH, exist_ok=True)

!pip install transformers -q

import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"디바이스: {device}")
print(f"GPU명: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# 나중에 런타임 끊겨도 복구 가능하도록 저장
# STEP 1에서 세 개 다 저장
df_train.to_csv(SAVE_PATH + 'train_set.csv', index=False)
df_val.to_csv(SAVE_PATH + 'val_set.csv', index=False)
df_test.to_csv(SAVE_PATH + 'test_set.csv', index=False)

print(f"✅ Train 저장 완료: {len(df_train):,}개")
print(f"✅ Val   저장 완료: {len(df_val):,}개")
print(f"✅ Test  저장 완료: {len(df_test):,}개")

In [ ]:
import pandas as pd
import json

# CSV 불러오기
df_train = pd.read_csv(SAVE_PATH + 'train_set.csv')
df_val   = pd.read_csv(SAVE_PATH + 'val_set.csv')
df_test  = pd.read_csv(SAVE_PATH + 'test_set.csv')


# 확인
print(f"✅ Train : {len(df_train):,}개")
print(f"✅ Val   : {len(df_val):,}개")
print(f"✅ Test  : {len(df_test):,}개")

In [ ]:
MODEL_NAME = "klue/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 64

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts  = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

BATCH_SIZE = 32

train_dataset = TextDataset(df_train['title_cleaned'], df_train['label'], tokenizer, MAX_LEN)
val_dataset   = TextDataset(df_val['title_cleaned'],   df_val['label'],   tokenizer, MAX_LEN)
test_dataset  = TextDataset(df_test['title_cleaned'],  df_test['label'],  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"✅ 데이터로더 완료")
print(f"Train 배치 수: {len(train_loader):,}")
print(f"Val   배치 수: {len(val_loader):,}")

In [ ]:
# categories 대신 이걸로 대체
num_labels = df_train['label'].nunique()
print(f"클래스 수: {num_labels}")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)
model =model.to(device)
print(f"✅ 모델 로드 완료 ({num_labels}개 클래스)")

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

EPOCHS      = 5
LR          = 2e-5

optimizer   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)
print(f"총 학습 스텝: {total_steps:,}")

In [ ]:
num_labels = len(categories)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)
model = model.to(device)
print(f"✅ 모델 로드 완료 ({num_labels}개 클래스)")

'''
# Class Weight (불균형 대응)
    class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(num_labels),
    y=df_train['label'].values
)
'''

loss_fn = nn.CrossEntropyLoss(
#    weight=class_weights_tensor,
    label_smoothing=0.1
)

EPOCHS = 5
LR     = 2e-5

optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)
print(f"총 학습 스텝: {total_steps:,}")

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for i, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds       = outputs.logits.argmax(dim=-1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

        if (i + 1) % 200 == 0:
            print(f"  [{i+1}/{len(loader)}] loss: {total_loss/(i+1):.4f} acc: {correct/total:.4f}")

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = loss_fn(outputs.logits, labels)

            total_loss += loss.item()
            preds       = outputs.logits.argmax(dim=-1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), correct / total, all_preds, all_labels

In [ ]:
# 학습 실행
best_val_loss     = float('inf')
patience          = 2
patience_counter  = 0

print("=" * 60)
print("🚀 학습 시작")
print("=" * 60)

for epoch in range(EPOCHS):
    print(f"\n[Epoch {epoch+1}/{EPOCHS}]")

    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, loss_fn, device
    )
    val_loss, val_acc, val_preds, val_labels = eval_epoch(
        model, val_loader, loss_fn, device
    )

    print(f"\n  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), SAVE_PATH + 'best_model.pt')
        print(f"  ✅ Best 모델 저장 (val_loss: {val_loss:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  ⚠️ 개선 없음 ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print("\n⛔ Early Stopping!")
            break

print("\n✅ 학습 완료!")

In [ ]:
# df_train에서 label 숫자 → 카테고리 이름 복구
# df_clean에서 매핑 가져오기
minor_cats = ['짝통', '중범죄', '안락사']

df_labeled = df_clean[df_clean['logic_tier'] != 'Tier 4'].copy()
df_labeled['category_final'] = df_labeled['final_category'].apply(
    lambda x: '기타' if x in minor_cats else x
)

categories = sorted(df_labeled['category_final'].unique())
cat2id     = {cat: i for i, cat in enumerate(categories)}
id2cat     = {i: cat for cat, i in cat2id.items()}

print(f"카테고리 수: {len(categories)}")
print(categories)

# 이번엔 꼭 저장
import json
with open(SAVE_PATH + 'cat2id.json', 'w', encoding='utf-8') as f:
    json.dump(cat2id, f, ensure_ascii=False)
print("✅ cat2id.json 저장 완료")

In [ ]:
# Best 모델 로드
model.load_state_dict(torch.load(SAVE_PATH + 'best_model.pt'))

_, test_acc, test_preds, test_labels_list = eval_epoch(
    model, test_loader, loss_fn, device
)

print("=" * 60)
print("📊 최종 Test 성능")
print("=" * 60)
print(f"전체 정확도: {test_acc:.4f}\n")

target_names = [id2cat[i] for i in range(num_labels)]
print(classification_report(
    test_labels_list, test_preds,
    target_names=target_names,
    digits=3
))

# 핵심 카테고리만 별도 출력
print("\n" + "=" * 60)
print("🎯 핵심 카테고리 성능")
print("=" * 60)
core_cats = ['유흥', '약물', '금융', '도박', '찌라시', '디비', '마약', '연락수단(텔레그램)']
core_ids  = [cat2id[c] for c in core_cats if c in cat2id]

core_true = [l for l in test_labels_list if l in core_ids]
core_pred = [p for l, p in zip(test_labels_list, test_preds) if l in core_ids]

print(classification_report(
    core_true, core_pred,
    labels=core_ids,
    target_names=[id2cat[i] for i in core_ids],
    digits=3
))


##### KoELECTRA 사용

### 지역명 매핑

In [ ]:
#df_clean = df_clean.drop(columns = 'region_list')

In [ ]:
import re
import unicodedata
import pandas as pd
from tqdm.auto import tqdm

# --- [준비 1] 시도 명칭 표준화 ---
sido_normalize = {
    '서울특별시': '서울', '경기도': '경기', '인천광역시': '인천', '강원도': '강원', '강원특별자치도': '강원',
    '충청북도': '충북', '충청남도': '충남', '전라북도': '전북', '전북특별자치도': '전북', '전라남도': '전남',
    '경상북도': '경북', '경상남도': '경남', '부산광역시': '부산', '대구광역시': '대구',
    '광주광역시': '광주', '대전광역시': '대전', '울산광역시': '울산', '세종특별자치시': '세종', '제주특별자치도': '제주'
}
broad_regions = set(sido_normalize.values())

# ---  우선순위 및 예외 사전 (문맥이 없을 때 사용)) ---
priority_map = {'서울': 1, '경기': 2, '인천': 3, '부산': 4, '경남': 5}
major_region_map = {'마산': '경남', '강서': '서울', '강서구': '서울', '합포': '경남', '회원': '경남'}

# --- [준비 3] 일반 용어 혼용 차단 사전 ---
high_risk_keywords = {
    '남성', '고기', '안내', '우리', '행복', '황금', '중앙', '복용', '신용', '유성', '문의',
    '비아', '사주', '가능', '달리', '대포', '통신', '하이', '자금', '유산', '장기', '다방',
    '중계', '정대', '하기', '동화', '동방', '회원', '안심', '시기', '정리', '안전', '초기',
    '자연', '흔적', '후기', '비용', '상태', '방법', '이후', '검사', '진행', '지수', '성지',
    '보안'
}

# ---------------------------------------------------------
# [단계 1] 역색인 사전(Inverted Map) 구축 (1:N 가능성 보존)
# ---------------------------------------------------------
inverted_region_map = {}

def add_to_map_final(kw, sido_full):
    kw = str(kw).strip()
    if not kw or len(kw) < 2: return
    # 시도 명칭 정규화 (서울특별시 -> 서울)
    sido_short = sido_normalize.get(sido_full, str(sido_full)[:2])
    if kw not in inverted_region_map:
        inverted_region_map[kw] = set()
    inverted_region_map[kw].add(sido_short)

# 행정동 리스트(df_admin)와 사용자 정의(df_r) 데이터를 모두 사전에 등록
for _, row in df_admin.iterrows():
    for target in [row['구'], row['동면리'], row['중분류_정답']]:
        add_to_map_final(target, row['시도'])

for _, row in df_r.iterrows():
    add_to_map_final(row['소분류'], row['대분류'])

# ---------------------------------------------------------
# [단계 2] 정밀 매핑 함수 (Context-Aware)
# ---------------------------------------------------------
def find_regions_precise(noun_text):
    if pd.isna(noun_text) or not str(noun_text).strip(): return "미지정"

    words = str(noun_text).replace(',', ' ').split()

    # 1. 문장에 직접 언급된 시도(Anchor) 식별
    mentioned_sidos = set()
    for w in words:
        # '경남' 같은 줄임말 혹은 '경상남도' 같은 풀네임 모두 체크
        norm_sido = sido_normalize.get(w, w[:2])
        if norm_sido in broad_regions:
            mentioned_sidos.add(norm_sido)

    # 2. 지명 후보군 수집 {지명: {시도1, 시도2}}
    found_candidates = {}
    for word in words:
        if word in inverted_region_map and word not in high_risk_keywords:
            found_candidates[word] = inverted_region_map[word]

    if not found_candidates:
        return "|".join(sorted(list(mentioned_sidos))) if mentioned_sidos else "미지정"

    # 3. 최종 매칭 결과 (시도:소분류 형식)
    final_pairs = set()

    for word, candidates in found_candidates.items():
        # [A] 문장에 언급된 시도가 있다면, 해당 시도와 일치하는 지명만 필터링
        if mentioned_sidos:
            match_with_context = candidates.intersection(mentioned_sidos)
            if match_with_context:
                for s in match_with_context:
                    final_pairs.add(f"{s}:{word}")
            # 언급된 시도와 겹치는 게 없으면, 노이즈로 간주하고 추가하지 않음 (강력한 필터링)
        else:
            # [B] 문장에 시도 언급이 없을 때 (우선순위/예외 사전 적용)
            if word in major_region_map:
                final_pairs.add(f"{major_region_map[word]}:{word}")
            else:
                best_sido = min(list(candidates), key=lambda x: priority_map.get(x, 99))
                final_pairs.add(f"{best_sido}:{word}")

    # 4. 직접 언급된 시도 중 짝이 없는 경우 추가
    matched_sidos = set([p.split(':')[0] for p in final_pairs])
    for s in mentioned_sidos:
        if s not in matched_sidos:
            final_pairs.add(f"{s}:{s}")

    unique_found = sorted(list(final_pairs))
    return "|".join(unique_found) if unique_found else "미지정"

# ---------------------------------------------------------
# [단계 3] 실행
# ---------------------------------------------------------
tqdm.pandas()
df_clean['region_list'] = df_clean['nouns'].progress_apply(find_regions_precise)

In [ ]:
import pandas as pd
from collections import Counter

# 1. 모든 명사의 출현 빈도 계산
all_nouns = [word for row in df_clean['nouns'].dropna() for word in str(row).split()]
noun_counts = Counter(all_nouns)

# 2. 사전에 등록된 지명 중, '블랙리스트에 없는' 것들만 필터링
new_region_audit = []
for small_cat, big_cat in inverted_region_map.items():
    # 이미 블랙리스트(high_risk_keywords)에 등록된 단어는 건너뜁니다!
    if small_cat in high_risk_keywords:
        continue

    if small_cat in noun_counts:
        new_region_audit.append({
            '지명(소분류)': small_cat,
            '지역(대분류)': big_cat,
            '전체 빈도': noun_counts[small_cat]
        })

# 3. 데이터프레임 변환 후 상위 30개 출력
df_new_audit = pd.DataFrame(new_region_audit).sort_values(by='전체 빈도', ascending=False)

print("🔍 [신규] 블랙리스트 제외, 오탐 의심 키워드 순위")
display(df_new_audit.head(50))

In [ ]:
# 1. 키워드가 포함된 행만 필터링
daejeon_check = df_clean[df_clean['nouns'].str.contains('수원', na=False)].copy()

print(f"🔎 키워드가 포함된 데이터 건수: {len(daejeon_check)}건")
print("-" * 50)

print("📂 [샘플 데이터 확인]")
display(daejeon_check[['title', 'nouns', 'region_list']].head(30))

from collections import Counter
daejeon_context = [word for row in daejeon_check['nouns'] for word in str(row).split() if word != '대전']
daejeon_top_words = Counter(daejeon_context).most_common(20)


### 저장

In [ ]:
df_clean.columns

In [ ]:
import os

save_dir = '/content/drive/MyDrive/크롤링 결과/'
file_name = '통합_불법광고_전처리_0502.csv'
save_path = os.path.join(save_dir, file_name)

# 한글 깨짐 방지를 위해 utf-8-sig 사용
df_clean.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"✅ 데이터셋 저장 완료! 경로: {save_path}")

# [분석 1] 워드 클라우드

*   전체 키워드 워드 클라우드
*   마약 카테고리 워드 클라우드
*   (추가) 마약 카테고리 주요 지역명 지도 시각화






In [ ]:
import pandas as pd
from google.colab import drive
import os

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. df_clean 저장된 파일 경로
file_path = '/content/drive/MyDrive/크롤링 결과/통합_불법광고_전처리_0225_tg_id_추가_중복X.csv'

# 3. 데이터 불러오기
if os.path.exists(file_path):
    df_clean = pd.read_csv(file_path, encoding='utf-8-sig')

    # 날짜 컬럼은 불러온 뒤 다시 datetime 타입으로 지정
    if 'date_dt' in df_clean.columns:
        df_clean['date_dt'] = pd.to_datetime(df_clean['date_dt'])

    print(f"✅ 데이터 로드 완료! (총 {len(df_clean):,}행)")
    # 중요한 컬럼들이 다 있는지 확인
    print(f"📊 포함된 컬럼: {list(df_clean.columns)}")
    display(df_clean.head(3))
else:
    print(f"❌ 파일을 찾을 수 없습니다. 경로가 맞는지 확인해주세요: {file_path}")

In [ ]:
import os
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from collections import Counter
import os

# 1. 폰트 재설치 (확실하게 하기 위해)
!apt-get update -qq
!apt-get install fonts-nanum -qq
!fc-cache -fv

# 2. 시스템에 설치된 나눔 폰트 경로 자동 검색
font_list = fm.findSystemFonts(fontpaths=None, fontext='ttf')
nanum_fonts = [f for f in font_list if 'Nanum' in f]

if nanum_fonts:
    # 가장 처음에 발견된 나눔 폰트를 사용
    FONT_PATH = nanum_fonts[0]
    print(f"✅ 사용 가능한 폰트를 찾았습니다: {FONT_PATH}")
else:
    # 폰트를 못 찾았을 경우 대비 (기본 폰트 경로)
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    print("⚠️ 나눔 폰트를 찾지 못해 기본 경로를 설정합니다.")

# 3. Matplotlib의 기본 폰트 설정 업데이트
font_name = fm.FontProperties(fname=FONT_PATH).get_name()
plt.rc('font', family=font_name)
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from wordcloud import WordCloud
from collections import Counter
import os


# 2. 제목용 폰트 설정
FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumSquareRoundR.ttf'
font_name = 'NanumSquareRound'
font_prop = fm.FontProperties(fname=FONT_PATH)

# 3. 워드클라우드 그리기 및 Top 10 추출 함수
def draw_korean_wc_with_top10(text_data, title, color_map):
    if not text_data or not str(text_data).strip():
        print(f"⚠️ '{title}' 데이터가 비어 있습니다.")
        return

    # 불용어 리스트
    stopwords = ['구입', '구매', '가격', '방법', '상담', '문의', '예약', '번호',
                 '주소', '사이트', '바로', '텔레', '텔레그램', '연락처', '가기',
                 '링크', '공지', '날짜', '판매', 'COM', 'com', '업체']

    # 명사 필터링 (2글자 이상 + 불용어 제외)
    words = [word for word in str(text_data).split() if word not in stopwords and len(word) > 1]
    word_counts = Counter(words)

    # --- Top 10 단어 출력 ---
    print(f"\n📊 [{title}] 빈도수 Top 10:")
    top_10 = word_counts.most_common(10)
    for i, (word, count) in enumerate(top_10, 1):
        print(f"{i}. {word}: {count}회")

    # --- 워드클라우드 생성 ---
    wc = WordCloud(
        font_path=FONT_PATH,
        background_color='white',
        width=1000,
        height=700,
        colormap=color_map,
        max_words=100,
        repeat=False
    )

    cloud = wc.generate_from_frequencies(word_counts)

    # --- 그래프 출력 ---
    plt.figure(figsize=(12, 9))
    plt.imshow(cloud, interpolation='bilinear')

    # 경로 직접 지정
    plt.title(f"[{title}] Word Cloud", fontproperties=font_prop, fontsize=25, pad=20)

    plt.axis('off')
    plt.show()

# --- 4. 실제 출력 ---

# [1] 전체 데이터 워드클라우드 및 Top 10
all_nouns = df_clean['nouns'].str.cat(sep=' ')
draw_korean_wc_with_top10(all_nouns, "전체 불법 광고", "viridis")

In [ ]:
df_kw['category'].unique()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from wordcloud import WordCloud
from collections import Counter
import random
import unicodedata
import pandas as pd

# =========================================
# [0] 사전 준비 (색상 함수 및 불용어)
# =========================================

# 폰트 경로 설정 (환경에 맞게 수정 필요)
# FONT_PATH = 'C:/Windows/Fonts/malgun.ttf' # 윈도우 예시
# FONT_PATH = '/System/Library/Fonts/AppleSDGothicNeo.ttc' # 맥 예시
# Colab의 경우: FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf' (설치 필요)

# 1. 색상 함수 정의 (빨강 / 파랑)
def red_color_func(word, font_size, position, orientation, random_state=None, **kwargs):
    return f"hsl({random.randint(0, 10)}, {random.randint(80, 100)}%, {random.randint(40, 50)}%)"

def blue_color_func(word, font_size, position, orientation, random_state=None, **kwargs):
    return f"hsl({random.randint(200, 240)}, {random.randint(60, 90)}%, {random.randint(45, 55)}%)"

# 2. 기본 노이즈 불용어 설정
base_stopwords = {
    '구입', '구매', '가격', '방법', '상담', '문의', '예약',
    '번호', '주소', '사이트', '바로', '텔레', '텔레그램',
    '연락처', '가기', '링크', '광고', '후기', '정보', '최신',
    '성인약국', '최고', '공식', '함유', '자이데나', '오케이',
    '효과', '상급', '구글', '독보', '전문', '마왕', '접속',
    '프릴리지', '좌표', '조회', '에포시스', '인기', '일등',
    '이순신', '행복', '고기', '퀵배송', '복용', '남성', '온라인', '대기', '달리'
}

# =========================================
# [1] 핵심 사전 구축 (df_r, df_kw 기반)
# =========================================

# 1-1. 지역명 사전 구축 (from df_r)
# 정규화 적용
target_regions = set([unicodedata.normalize('NFKC', str(r)) for r in df_r['소분류'].unique() if pd.notna(r)])

# 1-2. 마약 키워드 사전 구축 (from df_kw)
# 카테고리명이 '마약' 혹은 '약물' 등 실제 데이터에 맞게 수정하세요.
drug_categories = ['마약']
drug_dict_raw = df_kw[df_kw['category'].isin(drug_categories)]['keyword'].unique()
# 정규화 적용
drug_dictionary = set([unicodedata.normalize('NFKC', str(k)) for k in drug_dict_raw if pd.notna(k)])

print(f"✅ 사전 준비 완료: 지역 키워드 {len(target_regions)}개, 마약 키워드 {len(drug_dictionary)}개 로드됨.")


# =========================================
# [2] 그리기 메인 함수 정의
# =========================================

def draw_separated_wordclouds_v4(df):
    # --- 데이터 추출 및 필터링 ---
    print("데이터 추출 및 정제 중...")
    # 1. 'cat_마약'이 1인 행의 nouns만 추출해서 합침
    raw_text = df[df['cat_마약'] == 1]['nouns'].str.cat(sep=' ')

    # 2. 1차 필터링: 정규화 + 2글자 이상 + 기본 불용어 제외
    normalized_words = [unicodedata.normalize('NFKC', word) for word in raw_text.split()]
    filtered_words = [word for word in normalized_words if word not in base_stopwords and len(word) > 1]

    # --- 그룹 분리 (핵심 로직) ---
    # 📍 지역명: target_regions에 속하는 단어만 추출
    region_words = [word for word in filtered_words if word in target_regions]

    # 💊 마약 키워드: drug_dictionary에 속하는 단어만 추출
    drug_words = [word for word in filtered_words if word in drug_dictionary]

    # 빈도수 계산
    region_counts = Counter(region_words)
    drug_counts = Counter(drug_words)

    # --- 워드클라우드 생성 도우미 함수 ---
    def show_single_wc(counts, color_func, title, color_theme):
        if not counts:
            print(f"⚠️ [경고] '{title}' 데이터를 찾을 수 없습니다. 사전을 확인해주세요.")
            return

        wc = WordCloud(
            font_path=FONT_PATH,
            background_color='white',
            width=1000, height=800,
            max_words=100, # 단어 수 조정 가능
            relative_scaling=0.2,
            random_state=42
        ).generate_from_frequencies(counts).recolor(color_func=color_func)

        plt.figure(figsize=(12, 10))
        plt.imshow(wc, interpolation="bilinear")
        font_prop = fm.FontProperties(fname=FONT_PATH, size=25)
        plt.title(title, fontproperties=font_prop, color=color_theme, pad=20)
        plt.axis('off')
        plt.tight_layout()
        plt.show()

    # --- 개별 그리기 실행 ---
    print("🖥️ 1/2: 지역명(Red) 워드클라우드 생성 중...")
    show_single_wc(region_counts, red_color_func, "주요 타겟 지역 (지역명 사전 기반)", 'red')

    print("\n🖥️ 2/2: 마약 키워드(Blue) 워드클라우드 생성 중...")
    show_single_wc(drug_counts, blue_color_func, "핵심 마약 은어 (키워드 사전 기반)", 'blue')

# =========================================
# [3] 실행
# =========================================
# FONT_PATH가 올바르게 설정되었는지 꼭 확인하세요!
try:
    draw_separated_wordclouds_v4(df_clean)
except NameError:
     print("⚠️ FONT_PATH가 설정되지 않았습니다. 코드 상단에서 폰트 경로를 먼저 설정해주세요.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from wordcloud import WordCloud
from collections import Counter
import random

# --- [준비 단계] 리스트 및 데이터 정제 ---

# 1. 지역명 리스트 (df_r의 소분류 활용)
target_regions = set(df_r['소분류'].unique())

# 2. 사용자 요청 불용어 리스트 설정
base_stopwords = {
    '구입', '구매', '가격', '방법', '상담', '문의', '예약',
    '번호', '주소', '사이트', '바로', '텔레', '텔레그램',
    '연락처', '가기', '링크', '광고', '후기', '정보', '최신',
    '성인약국', '최고', '공식', '함유', '자이데나', '오케이',
    '효과', '상급', '구글', '독보', '전문', '마왕', '접속',
    '프릴리지', '좌표', '조회', '에포시스', '인기', '일등',
    '이순신', '판매'
}

# 3. 약물 카테고리 키워드 추가 (df_kw에서 추출된 drug_stopwords 포함)
drug_categories = ['마약'] # 혹은 df_kw['category'].unique() 중 마약 관련 항목들
drug_stopwords = df_kw[df_kw['category'].isin(drug_categories)]['keyword'].unique()

# 정규화(NFKC)를 적용하여 nouns 컬럼과 형식을 맞춥니다.
drug_dictionary = set([unicodedata.normalize('NFKC', str(k)) for k in drug_stopwords])

final_stopwords = base_stopwords.union(drug_stopwords)

# 4. 색상 함수 정의: 지역명 리스트에 있는 단어만 정확히 빨간색
def integrated_color_func(word, font_size, position, orientation, random_state=None, **kwargs):
    if word in target_regions:
        # 지역명: 빨간색 계열
        return f"hsl({random.randint(0, 5)}, {random.randint(85, 100)}%, {random.randint(45, 55)}%)"
    else:
        # 그 외(마약 관련): 파란색 계열
        return f"hsl({random.randint(200, 240)}, {random.randint(60, 90)}%, {random.randint(40, 60)}%)"

# 5. 워드클라우드 실행 함수
def draw_integrated_drug_wc(df, title):
    # 'cat_마약' 데이터에서 명사 추출
    raw_text = df[df['cat_마약'] == 1]['nouns'].str.cat(sep=' ')

    # 단어 필터링: 2글자 이상 + 지정된 final_stopwords 제외
    words = [word for word in raw_text.split() if word not in final_stopwords and len(word) > 1]
    word_counts = Counter(words)

    if not word_counts:
        print("⚠️ 설정된 불용어를 제외한 후 분석할 데이터가 없습니다.")
        return

    # 워드클라우드 객체 생성
    wc = WordCloud(
        font_path=FONT_PATH,
        background_color='white',
        width=1200,
        height=800,
        max_words=150,
        relative_scaling=0.3,
        random_state=42
    ).generate_from_frequencies(word_counts)

    # 출력
    plt.figure(figsize=(15, 10))
    # recolor를 통해 지역명만 빨간색으로 입힘
    plt.imshow(wc.recolor(color_func=integrated_color_func), interpolation='bilinear')

    font_prop = fm.FontProperties(fname=FONT_PATH, size=25)
    plt.title(f"[{title}] 지역명(Red) vs 마약키워드(Blue)", fontproperties=font_prop, pad=30)
    plt.axis('off')
    plt.show()

# 실행
draw_integrated_drug_wc(df_clean, "마약 매크로 통합 모니터링")

# [분석 2] 마약 종류별 지역별 분포

In [ ]:
import pandas as pd

# 1. 구글 시트 URL 정보 추출
# URL: https://docs.google.com/spreadsheets/d/1KXU__1wXg-oY1vJWxy7HkOhrZDe_v8NGxLn8MacQC9M/edit?gid=454780331#gid=454780331
sheet_id = "1KXU__1wXg-oY1vJWxy7HkOhrZDe_v8NGxLn8MacQC9M"
gid = "454780331" # 특정 탭을 불러오기 위한 GID
export_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

# 2. 데이터 로드
try:
    df_drug_sheet = pd.read_csv(export_url)
    print("✅ 마약 키워드 분류 시트 로드 성공!")

    # 3. 데이터 기본 정제 (공백 제거 등)
    # 컬럼명이 '마약대분류', '마약소분류', '마약키워드'인지 확인합니다.
    df_drug_sheet.columns = [c.strip() for c in df_drug_sheet.columns]

    display(df_drug_sheet.head())

except Exception as e:
    print(f"❌ 로드 실패: 시트가 '링크가 있는 모든 사용자에게 공개' 상태인지 확인해주세요.")
    print(f"에러 메시지: {e}")

In [ ]:
import pandas as pd
from collections import Counter
import unicodedata

# 1. 시트 데이터(df_drug_sheet) 전처리: 키워드 펼치기
# 시트 데이터가 df_drug_sheet라는 이름으로 로드되어 있다고 가정합니다.
drug_data = []
for _, row in df_drug_sheet.iterrows():
    major = row['마약대분류']
    minor = row['마약소분류']
    # 쉼표로 구분된 키워드들을 분리하고 공백 제거 및 정규화
    keywords = [unicodedata.normalize('NFKC', k.strip()) for k in str(row['마약키워드']).split(',') if k.strip()]

    for kw in keywords:
        drug_data.append({'대분류': major, '소분류': minor, '키워드': kw})

# 키워드 매핑용 데이터프레임 생성
df_drug_map = pd.DataFrame(drug_data)

# 2. df_clean에서 마약 관련 행만 추출 및 텍스트 통합
# 이미 분류된 'cat_마약' 컬럼을 활용합니다.
df_drug_ads = df_clean[df_clean['cat_마약'] == 1].copy()
all_drug_nouns = " ".join(df_drug_ads['nouns'].fillna("")).split()
word_counts = Counter(all_drug_nouns)

# 3. 키워드별 빈도 매핑
df_drug_map['빈도'] = df_drug_map['키워드'].apply(lambda x: word_counts.get(x, 0))

# 4. 결과 요약: 소분류 및 대분류별 통계
# 소분류별 합계
minor_summary = df_drug_map.groupby(['대분류', '소분류'])['빈도'].sum().reset_index().sort_values(by='빈도', ascending=False)

# 대분류별 합계
major_summary = df_drug_map.groupby('대분류')['빈도'].sum().reset_index().sort_values(by='빈도', ascending=False)

# 5. 결과 출력
print("📊 [마약 대분류별] 등장 빈도 요약")
display(major_summary)

print("\n📊 [마약 소분류별] 등장 빈도 TOP 10")
display(minor_summary.head(10))

print("\n🔍 [개별 키워드별] 실제 매칭 결과 (빈도 높은 순)")
display(df_drug_map[df_drug_map['빈도'] > 0].sort_values(by='빈도', ascending=False).head(20))

In [ ]:
import pandas as pd
from collections import Counter
import unicodedata

# 1. 분석 대상 데이터 필터링 (cat_마약 == 1)
df_drug_ads = df_clean[df_clean['cat_마약'] == 1].copy()

# 2. 통합 분석 함수 (원본 텍스트에서 대/소분류 매핑)
def get_drug_info_from_map(text, mapping_df):
    if pd.isna(text): return []
    # 텍스트 정규화 (NFKC)
    target_text = unicodedata.normalize('NFKC', str(text)).strip()

    found_items = []
    # df_drug_map의 각 행을 순회하며 키워드 매칭
    for _, row in mapping_df.iterrows():
        kw = row['키워드']
        if kw in target_text:
            # (대분류, 소분류) 쌍으로 저장
            found_items.append((row['대분류'], row['소분류']))

    return list(set(found_items)) # 한 게시글 내 중복 매칭 제거

# 3. 데이터 평탄화 (Explode) 처리
hierarchy_results = []

target_text_col = 'nouns'

for _, row in df_drug_ads.iterrows():
    # [A] 지역 정보 추출 (region_list 활용)
    # region_list가 ['서울:강남', '경기:수원'] 형태의 리스트라고 가정
    regions = row['region_list']
    if not isinstance(regions, list):
        regions = str(regions).split('|') # 문자열일 경우 대비

    sidos = set()
    for r in regions:
        if ':' in r:
            sidos.add(r.split(':')[0]) # '서울'만 추출
        elif r and r != "미지정":
            sidos.add(r)

    if not sidos: sidos = {'미지정'}

    # [B] 마약 계층 정보 추출 (원본 텍스트 사용!)
    drug_items = get_drug_info_from_map(row[target_text_col], df_drug_map)

    if not drug_items:
        drug_items = [('미식별', '미식별')]

    # [C] 모든 조합 생성
    for sido in sidos:
        for major, minor in drug_items:
            hierarchy_results.append({
                '지역': sido,
                '마약대분류': major,
                '마약소분류': minor
            })

# 4. 최종 데이터프레임 생성 및 집계
df_final_stat = pd.DataFrame(hierarchy_results)
df_grouped = df_final_stat.groupby(['지역', '마약대분류', '마약소분류']).size().reset_index(name='검출건수')

# 5. 결과 정렬 (지역별, 검출건수 많은 순)
df_grouped = df_grouped.sort_values(by=['지역', '검출건수'], ascending=[True, False])

print("📊 [지역별 마약 대분류/소분류 유통 통계]")
display(df_grouped)

In [ ]:
# 5. 지역별 상위 4개 소분류 추출 및 정렬
# 이미 df_grouped가 '지역' 오름차순, '검출건수' 내림차순으로 정렬되어 있는 상태입니다.
df_top4 = df_grouped.groupby('지역').head(4).reset_index(drop=True)

print("📊 [지역별 마약 소분류 유통 통계 - 지역별 상위 4개 추출]")
display(df_top4)

# (선택 사항) 만약 결과를 가독성 있게 피벗 테이블 형태로 보고 싶다면 아래 코드를 사용하세요.
# df_pivot = df_top4.pivot_table(index=['지역', '마약대분류'], columns='마약소분류', values='검출건수').fillna(0)
# display(df_pivot)

In [ ]:
# 1. '미식별' 데이터 제외
df_filtered = df_grouped[df_grouped['마약소분류'] != '미식별'].copy()

# 2. 지역별로 검출건수 상위 4개 추출
# (이미 df_grouped가 정렬되어 있으므로 head(4) 사용)
df_top4_no_unknown = df_filtered.groupby('지역').head(4)

# 3. 피벗 테이블 생성 (행: 지역, 열: 마약소분류, 값: 검출건수)
df_pivot = df_top4_no_unknown.pivot_table(
    index='지역',
    columns='마약소분류',
    values='검출건수',
    aggfunc='sum'
).fillna(0).astype(int)

# 4. 가독성을 위해 전체 검출 합계가 높은 지역 순으로 정렬
df_pivot['지역별_합계'] = df_pivot.sum(axis=1)
df_pivot = df_pivot.sort_values(by='지역별_합계', ascending=False).drop(columns='지역별_합계')

print("📊 [지역별 마약 소분류 TOP 4 현황 (미식별 제외)]")
display(df_pivot)

In [ ]:
# 1. 분석 대상 소분류 리스트 정의
target_drugs = ['필로폰', '엑스터시', '암페타민', '코카인']

# 2. 지정된 소분류에 해당하는 데이터만 필터링
df_specific = df_grouped[df_grouped['마약소분류'].isin(target_drugs)].copy()

# 3. 피벗 테이블 생성 (행: 지역, 열: 마약소분류, 값: 검출건수)
df_pivot = df_specific.pivot_table(
    index='지역',
    columns='마약소분류',
    values='검출건수',
    aggfunc='sum'
).fillna(0).astype(int)

# 4. 분석 대상 4개 컬럼이 모두 표시되도록 순서 보장 (데이터가 없는 컬럼도 0으로 표시)
for drug in target_drugs:
    if drug not in df_pivot.columns:
        df_pivot[drug] = 0

# 컬럼 순서 고정
df_pivot = df_pivot[target_drugs]

# 5. 지역별 합계 기준 내림차순 정렬
df_pivot['지역별_합계'] = df_pivot.sum(axis=1)
df_pivot = df_pivot.sort_values(by='지역별_합계', ascending=False).drop(columns='지역별_합계')

print("📊 [지역별 주요 4종 마약 유통 현황]")
display(df_pivot)

In [ ]:
# 1. 분석 대상 소분류 리스트 정의
target_drugs = ['필로폰', '엑스터시', '암페타민', '코카인']

# 2. 지정된 소분류에 해당하는 데이터만 필터링
df_specific = df_grouped[df_grouped['마약소분류'].isin(target_drugs)].copy()

# 3. 피벗 테이블 생성 (행: 지역, 열: 마약소분류, 값: 검출건수)
df_pivot = df_specific.pivot_table(
    index='지역',
    columns='마약소분류',
    values='검출건수',
    aggfunc='sum'
).fillna(0).astype(int)

# 4. 분석 대상 4개 컬럼이 모두 표시되도록 순서 보장 (데이터가 없는 컬럼도 0으로 표시)
for drug in target_drugs:
    if drug not in df_pivot.columns:
        df_pivot[drug] = 0

# 컬럼 순서 고정
df_pivot = df_pivot[target_drugs]

# 1~4번 과정은 동일하게 수행되었다고 가정합니다.

# 5. '미지정' 행 제거
if '미지정' in df_pivot.index:
    df_pivot = df_pivot.drop(index='미지정')

# 6. '전국' 행 처리: 존재한다면 따로 분리했다가 나중에 합침
if '전국' in df_pivot.index:
    # '전국'을 제외한 데이터들로 합계 기준 내림차순 정렬
    df_others = df_pivot.drop(index='전국')
    df_others['total'] = df_others.sum(axis=1)
    df_others = df_others.sort_values(by='total', ascending=False).drop(columns='total')

    # '전국' 행만 따로 추출
    df_national = df_pivot.loc[['전국']]

    # 순서대로 병합 (정렬된 지역들 + 마지막에 전국)
    df_pivot = pd.concat([df_others, df_national])
else:
    # '전국'이 없는 경우 일반적인 내림차순 정렬만 수행
    df_pivot['total'] = df_pivot.sum(axis=1)
    df_pivot = df_pivot.sort_values(by='total', ascending=False).drop(columns='total')

print("📊 [지역별 주요 4종 마약 유통 현황]")
display(df_pivot)

# [분석 3] 텔레그램 아이디 네트워크 분석




In [ ]:
#df_clean = df_clean.drop(columns = 'tg_id')

In [ ]:
import re
import pandas as pd
import unicodedata
from tqdm.auto import tqdm

# ---------------------------------------------------------
# [단계 1] 환경 설정 및 노이즈 필터 정의
# ---------------------------------------------------------
tg_keywords = ['텔레그램', '텔ㄹ그ㄹ', '텔ㄹㄱㄹ', '텔레', '텔램', '탥래', '탤그', '텕', '텔', 'tg']

# 아이디가 아닌 일반 홍보 단어
tg_noise_words = {
    'club', 'canalis', 'kakaotalkid', 'beer', 'barbeerso6nguyenhoang',
    'info', 'check', 'admin', 'kakao', 'line', 'link', 'video', 'photo'
}

def is_valid_tg_id(id_str):
    """아이디 유효성 검증: 길이(4~24), 시작문자(영문), 홍보단어"""
    id_str = id_str.strip().lower()
    if not (4 <= len(id_str) <= 24): return False
    if not re.match(r'^[a-z]', id_str): return False
    if id_str in tg_noise_words: return False
    if any(noise in id_str for noise in ['kakaotalk', 'kakao', 'admin']): return False
    return True

# ---------------------------------------------------------
# [단계 2] 추출 함수 (포함 관계 정리 로직 제거)
# ---------------------------------------------------------
def extract_tg_id_final(text):
    if pd.isna(text) or not str(text).strip(): return ""

    # 정규식 패턴 (키워드 기반 + @ 기반)
    tg_pattern = re.compile(r'(?i)(?:' + '|'.join(re.escape(k) for k in tg_keywords) + r')[:\s@]*([a-zA-Z0-9_]{4,})')
    at_pattern = re.compile(r'(?:^|\s)@([a-zA-Z0-9_]{4,})')

    # 1. 후보군 추출 및 소문자화 (중복 제거를 위해 set 사용)
    raw_candidates = set([c.lower() for c in (tg_pattern.findall(text) + at_pattern.findall(text))])

    # 2. 유효성 검증만 수행 (포함 관계 삭제 없이 모두 유지)
    valid_ids = [c for c in raw_candidates if is_valid_tg_id(c)]

    # 추출된 결과들을 | 로 연결하여 반환
    return "|".join(sorted(valid_ids))

print("🔍 텔레그램 아이디 추출 중...")
tqdm.pandas()
df_clean['tg_id'] = df_clean['title'].progress_apply(extract_tg_id_final)


In [ ]:
# 1. '|'를 기준으로 리스트로 만든 뒤, 각 요소를 개별 행으로 펼칩니다.
exploded_ids = df_clean['tg_id'].str.split('|').explode()

# 2. 공백이나 빈 문자열 등을 제거합니다.
exploded_ids = exploded_ids[exploded_ids.str.strip() != ""]

# 3. 고유한 아이디만 추출하여 개수를 확인합니다.
unique_id_list = exploded_ids.unique()
total_count = len(unique_id_list)

print(f"전체 고유 텔레그램 아이디 개수: {total_count}개")

In [ ]:
# 1. '|'가 포함된 유니크한 '패턴'들만 추출
multi_patterns = [x for x in df_clean['tg_id'].unique() if '|' in str(x)]

print(f"🧐 '|'가 포함된 유니크한 패턴 개수: {len(multi_patterns)}개")
print(f"📝 실제 패턴 예시: {multi_patterns}")

# 2. '|'가 포함된 전체 '행(Row)'의 개수 확인
multi_row_count = df_clean['tg_id'].str.contains('|', regex=False, na=False).sum()
print(f"📢 '|'가 포함된 전체 게시글 수: {multi_row_count}개")

In [ ]:
unique_id_list

In [ ]:
# 1. 아이디별 행 분리 (Explode)
# 여기서 'cupid4989|xinxiandb'가 2개의 행으로 쪼개져 각각 통계에 반영됩니다.
df_tg = df_clean[df_clean['tg_id'] != ""].copy()
df_tg['tg_id_single'] = df_tg['tg_id'].str.split('|')
df_exploded = df_tg.explode('tg_id_single')

# 2. cat_으로 시작하는 범죄 카테고리 자동 탐색
crime_categories = [
    col for col in df_clean.columns
    if col.startswith('cat_') and col != 'cat_연락수단'
]

# 3. 아이디별 범죄 가담 현황 통계
tg_crime_stats = df_exploded.groupby('tg_id_single')[crime_categories].sum()
tg_crime_stats['관여_카테고리_수'] = (tg_crime_stats[crime_categories] > 0).sum(axis=1)

# 4. 결과 정렬
tg_crime_stats = tg_crime_stats.sort_values(by=['관여_카테고리_수'], ascending=False)

print(f"✨ 분석 완료! 총 {len(crime_categories)}개의 카테고리에 대해 아이디별 통계를 산출했습니다.")
display(tg_crime_stats.head(20))

In [ ]:
# 1. 'cat_'으로 시작하되 '연락수단'은 제외한 컬럼 리스트 생성
crime_categories = [col for col in df_clean.columns
                    if col.startswith('cat_') and col != 'cat_연락수단']

# 2. 카테고리별 활용률 계산
category_tg_usage = []

for cat in crime_categories:
    # 해당 카테고리의 전체 게시글 수
    total_posts = len(df_clean[df_clean[cat] == 1])
    # 해당 카테고리 중 텔레그램 아이디가 하나라도 추출된 게시글 수
    tg_posts = len(df_clean[(df_clean[cat] == 1) & (df_clean['tg_id'] != "")])

    usage_rate = (tg_posts / total_posts) * 100 if total_posts > 0 else 0

    category_tg_usage.append({
        '카테고리': cat.replace('cat_', ''), # 'cat_' 접두사 제거하여 가독성 향상
        '전체_게시글': total_posts,
        '텔레그램_노출_글': tg_posts,
        '텔레그램_활용률(%)': round(usage_rate, 2)
    })

# 3. 데이터프레임 변환 및 정렬
df_usage_summary = pd.DataFrame(category_tg_usage).sort_values(by='텔레그램_노출_글', ascending=False)

print("📊 [범죄 카테고리별 텔레그램 활용 현황 (연락수단 제외)]")
display(df_usage_summary)

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import community.community_louvain as community_louvain

# 1. '연락수단' 제외 및 데이터 집계 (기존 로직 유지)
cat_columns = [col for col in df_tg.columns if col.startswith('cat_') and col != 'cat_연락수단']

# 사이트별 카테고리별 글 개수 합산
site_sums = df_tg.groupby('site')[cat_columns].sum()
# 아무 카테고리에도 해당하지 않는 사이트 제외
site_sums = site_sums[site_sums.sum(axis=1) > 0]

# 각 사이트별 '주력 카테고리'의 글 개수(최댓값) 추출 -> 노드 크기용
site_main_cat_count = site_sums.max(axis=1)
# 각 사이트별 주력 카테고리 명칭 추출 -> 색상용
site_info = site_sums.idxmax(axis=1)

valid_sites = site_info.index.tolist()
df_filtered = df_tg[df_tg['site'].isin(valid_sites)]

# 2. 그래프 생성
G = nx.Graph()
for tid, group in df_filtered.groupby('tg_id'):
    sites = group['site'].unique()
    if len(sites) > 1:
        for i in range(len(sites)):
            for j in range(i + 1, len(sites)):
                current_weight = G.get_edge_data(sites[i], sites[j], {'weight': 0})['weight']
                G.add_edge(sites[i], sites[j], weight=current_weight + 1)

# 3. Louvain 군집 탐색
partition = community_louvain.best_partition(G)
unique_clusters = set(partition.values())

# 4. 군집별 확대 시각화
for cluster_id in unique_clusters:
    nodes_in_cluster = [node for node, clus in partition.items() if clus == cluster_id]

    if len(nodes_in_cluster) < 3:
        continue

    subgraph = G.subgraph(nodes_in_cluster)

    plt.figure(figsize=(12, 10))
    pos = nx.spring_layout(subgraph, k=0.8, seed=42)

    # [수정 포인트] 노드 크기 설정
    # 주력 카테고리 글 수에 비례하게 설정 (너무 작거나 크지 않게 스케일링)
    # 기본 크기 300 + (글 개수 * 배율)
    node_sizes = [300 + (site_main_cat_count[node] * 0.3) for node in subgraph.nodes()]

    # 색상 결정
    cluster_node_colors = [color_map_dict[site_info[node]] for node in subgraph.nodes()]

    # 그리기
    nx.draw_networkx_edges(subgraph, pos, alpha=0.3, edge_color='gray',
                           width=[d['weight']*0.5 for u, v, d in subgraph.edges(data=True)])

    nx.draw_networkx_nodes(subgraph, pos,
                           node_size=node_sizes,
                           node_color=cluster_node_colors,
                           alpha=0.8)

    # 라벨 위치 조정 (노드 크기에 따라 유동적으로 조정)
    label_pos = {node: (coords[0], coords[1] + 0.1) for node, coords in pos.items()}
    nx.draw_networkx_labels(subgraph, label_pos, font_size=11,
                            font_family=font_name, font_weight='bold')

    plt.title(f"Cluster {cluster_id} 상세 분석 (노드 크기: 주력 카테고리 빈도)", fontsize=15)
    plt.axis('off')
    plt.show()

# [분석 4] DB 세부 카테고리 분석

*   뿡자네 제주귤밭 한정



In [ ]:
import pandas as pd
import re

# 1) 사이트 필터
df_p = df_clean.loc[df_clean["site"] == "뿡자네 제주귤밭"].copy()

# 2) "DB/디비/db" 앞에 붙는 단어 추출 패턴
# - 예: "회원DB", "회원 DB", "고객디비", "고객 db"

pat = re.compile(r'([가-힣0-9]+)\s*(?:DB|db|디비)\b')

# 3) title_cleaned에서 추출
m = (
    df_p["nouns"]
    .fillna("")
    .astype(str)
    .str.extractall(pat)   # 결과: MultiIndex (row, match_no)
    .reset_index(level=1, drop=True)[0]  # 0번 그룹(앞 단어)만
)

# 4) 통계: 빈도, 비율
prefix_counts = m.value_counts().rename_axis("prefix").reset_index(name="count")
prefix_counts["pct"] = (prefix_counts["count"] / prefix_counts["count"].sum() * 100).round(2)

# 상위 20개 확인
prefix_counts.head(11)

In [ ]:
# 1. 나눔 폰트 설치
!apt-get update -qq
!apt-get install fonts-nanum -qq
#/usr/share/fonts/truetype/nanum/NanumGothic.ttf

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 1. 폰트 경로 설정 (코랩 설치 기본 경로)
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
font_prop = fm.FontProperties(fname=font_path, size=12)
title_prop = fm.FontProperties(fname=font_path, size=18, weight='bold')

# 2. 데이터 준비 (상위 10개 추출)
df_top10_p = prefix_counts.head(10)

# 3. 그래프 그리기
plt.figure(figsize=(12, 7))
# 색상을 오렌지 계열로 변경 (제주귤밭 컨셉)
bars = plt.bar(df_top10_p['prefix'], df_top10_p['count'], color='darkorange', alpha=0.8, edgecolor='black')

# 4. 한글 폰트 개별 적용
plt.title("[뿡자네 제주귤밭] DB 키워드 TOP 10 분석", fontproperties=title_prop, pad=20)
plt.xlabel("키워드", fontproperties=font_prop)
plt.ylabel("글 개수 (건)", fontproperties=font_prop)

# 5. X축 레이블(키워드명) 한글 깨짐 방지
plt.xticks(range(len(df_top10_p)), df_top10_p['prefix'], fontproperties=font_prop, rotation=45)

# 6. 막대 위에 숫자 표시
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.1,
             f'{int(height)}', ha='center', va='bottom', fontproperties=font_prop)

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import re

# 전체 데이터프레임 기준 DB 카테고리
pat = re.compile(r'([가-힣0-9]+)\s*(?:DB|db|디비)\b')

m = (
    df_clean["nouns"]
    .fillna("")
    .astype(str)
    .str.extractall(pat)
    .reset_index(level=1, drop=True)[0]  # 0번 그룹(앞 단어)만
)

# 4) 통계: 빈도, 비율
prefix_counts = m.value_counts().rename_axis("prefix").reset_index(name="count")
prefix_counts["pct"] = (prefix_counts["count"] / prefix_counts["count"].sum() * 100).round(2)

# 상위 20개 확인
prefix_counts.head(20)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 1. 폰트 경로 설정 (코랩 기본 설치 경로)
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
font_prop = fm.FontProperties(fname=font_path, size=12)

# 2. 데이터 준비 (상위 10개)
df_top10 = prefix_counts.head(10)

# 3. 그래프 그리기
plt.figure(figsize=(12, 7))
bars = plt.bar(df_top10['prefix'], df_top10['count'], color='dodgerblue')

# 4. 한글 적용 (직접 주입 방식)
plt.title("[전체 사이트] DB 관련 키워드 TOP 10 빈도수 분석", fontproperties=fm.FontProperties(fname=font_path, size=18), pad=20)
plt.xticks(range(len(df_top10)), df_top10['prefix'], fontproperties=font_prop, rotation=45)
plt.ylabel("글 개수 (건)", fontproperties=font_prop)

# 막대 위 숫자 표시
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.5, f'{int(height)}', ha='center', fontproperties=font_prop)

plt.tight_layout()
plt.show()

# [분석 5] 연락수단 파이차트

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 1. 폰트 설정 적용
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
font_prop = fm.FontProperties(fname=font_path, size=12)
plt.rc('font', family='NanumGothic') # 전역 폰트 설정

# 2. 카테고리별 키워드 정의
keywords_map = {
    '텔레그램': ['텔레', '텔램', '텔레그램', '탥래', '텔', 'tg', '탤그', '텔ㄹ그ㄹ', '텔ㄹㄱㄹ', '텕'],
    '카카오톡': ['톡', '카톡', '카카오톡', 'ㅋ ㅏ톡', '까똑'],
    '전화번호': ['010', '01ㅇ', '0!0', 'O!O', '0IO', 'olo', 'oIo', 'o1o', 'O 1 O', '+8210', 'Ol0', 'OIO', '060', '070']
}

# 3. 빈도수 집계 (nouns 컬럼 대상)
# nouns 내부에 키워드가 포함되어 있는지 확인하여 카운트
category_counts = {cat: 0 for cat in keywords_map.keys()}

for text_data in df_clean['nouns'].astype(str):
    for category, keywords in keywords_map.items():
        # 해당 카테고리의 키워드 중 하나라도 텍스트에 포함되어 있다면 1회 가산
        if any(kw in text_data for kw in keywords):
            category_counts[category] += 1

# 4. 시각화 데이터 준비
plot_df = pd.Series(category_counts)

# 5. 파이차트 생성
if plot_df.sum() == 0:
    print("⚠️ 'nouns' 컬럼에서 매칭된 키워드가 없습니다. 데이터를 확인해주세요.")
else:
    plt.figure(figsize=(10, 8), facecolor='white')

    colors = ['#0088cc', '#00c300', '#fee500', '#ff5a5f'] # 텔레/라인/카톡/전화번호 상징색

    # 파이차트 그리기
    wedges, texts, autotexts = plt.pie(
        plot_df,
        labels=plot_df.index,
        autopct='%1.1f%%',
        startangle=140,
        colors=colors,
        explode=[0.05 if i == plot_df.argmax() else 0 for i in range(len(plot_df))],
        shadow=True
    )

    # 폰트 속성 수동 적용 (각 텍스트 요소에 적용)
    for t in texts:
        t.set_fontproperties(font_prop)
    for at in autotexts:
        at.set_color('black')
        at.set_weight('bold')

    plt.title('연락수단별 노출 비중 (nouns 분석 기반)', fontproperties=font_prop, fontsize=16, pad=20)
    plt.show()

# 집계된 실제 수치 확인
print("--- 집계 상세 결과 ---")
print(plot_df)